<h3>Single image processing (with napari visualization)</h3>

### Import dependencies and select GPU device
Load utility functions and analysis libraries required for segmentation, quantification, and visualization.

In [42]:
from utils import extract_scaling_metadata, predict_nuclei_labels, simulate_cytoplasm, map_df_column_to_labels
from io_utils import list_images, ensure_output_dir, load_precomputed_results_if_available
from pathlib import Path
import tifffile
import czifile
import napari
import numpy as np
from skimage.measure import regionprops_table
import pandas as pd
import pyclesperanto_prototype as cle
import plotly.express as px

cle.select_device("RTX")

<NVIDIA GeForce RTX 4090 Laptop GPU on Platform: NVIDIA CUDA (2 refs)>

### Configure experiment inputs
Define data location and per channel info used in the workflow.

Input needed from researcher: update RAW_DATA_DIRECTORY, NUCLEI_CHANNEL, MIN_MAX_NUCLEI_VOLUME, channel indices, marker definitions, and index for the image to explore from your input directory.

In [43]:
# Copy the path where your .czi files are stored, you can use absolute or relative paths to point at other disk locations
RAW_DATA_DIRECTORY = "./raw_data/WT_organoids/"
#RAW_DATA_DIRECTORY = "./raw_data/20240703 PGE2MSN Tum_organoids"

# Channel index used for CellposeSAM-based 3D nuclei segmentation
NUCLEI_CHANNEL = 2

# Minimum and maximum nuclei label volume to use for filtering predicted nuclei labels (min, max) i.e (250, 4000)
# If set to None, no filtering is applied and all predicted labels are returned
MIN_MAX_NUCLEI_VOLUME = (250, 4000)

# Cytoplasm thickness in voxels (starting from the nucleus)
# If padding is applied, the thickness will be applied to the padded nucleus
CYTOPLASM_THICKNESS = 1

# Padding to keep a safe distance between nuclei and cytoplasm borders
# It expands the nucleus by the padding value in all directions
NUCLEI_PADDING = 1

# List of markers used in the experiment, each with its corresponding channel index
MARKERS = [("UEA-1", 0), ("YAP", 1), ("Nuclei", 2), ("BCAT", 3)]

# Mark the position of the .czi iamge you want to open in your raw data folder (first = 0, second = 1, third = 2)
CZI_IMAGE_INDEX = 1

### Load selected image and parse identifiers
Read the chosen `.czi` file, squeeze dimensions, and extract experiment metadata from filename fields.

Input needed from researcher: ensure your filename format supports experiment, mouse, treatment, and replica parsing.

In [44]:
directory_path = Path(RAW_DATA_DIRECTORY)
input_folder_id = directory_path.stem
images = list_images(directory_path, "czi")

image = images[CZI_IMAGE_INDEX]

# Read path storing raw image and extract filename
file_path = Path(image)
filename = file_path.stem

# Read the image file and remove singleton dimensions
img = czifile.imread(image)
img = img.squeeze()

# Extract experiment, mouse, treatment and replica ids
experiment_id = filename.split(" ")[0]
mouse_id = filename.split(" ")[1]
treatment_id = filename.split(" ")[2]
replica_id = filename.split(" ")[-1]


### Launch 3D napari viewer
Open the raw channels in napari for visual quality control before segmentation and quantification.

Input needed from researcher: verify channel names, colors, and orientation are correct.

In [45]:
# Initialize Napari Viewer and display all input channels
viewer = napari.Viewer(ndisplay=3)
viewer.add_image(img, 
                channel_axis=0,
                colormap=['white', 'magenta', 'cyan', 'yellow'],
                name=[tuple[0] for tuple in MARKERS]
                )

[<Image layer 'UEA-1' at 0x1ed8d513790>,
 <Image layer 'YAP' at 0x1ed8d53f2b0>,
 <Image layer 'Nuclei' at 0x1ed8cd3aef0>,
 <Image layer 'BCAT' at 0x1ed8d5bf9a0>]

### Nuclei label prediction using CellposeSAM & cytoplasm simulation
This step loads existing nuclei labels if present, or predicts and saves new labels.
Then based on those nuclei labels it performs a morphological dilation operation to simulate the surrounding cytoplasm

In [46]:
# Ensure output directory for this container's nuclei labels
nuclei_labels_dir = ensure_output_dir(RAW_DATA_DIRECTORY, input_folder_id, results_type="nuclei_labels")
print(f"Nuclei labels directory: {nuclei_labels_dir}")

# Calculate anisotropy CellposeSAM parameter to rescale across the Z-axis (ratio of Z-resolution to XY-resolution)
# Extract x,y,z scaling from .czi file metadata
scaling_x_um, scaling_y_um, scaling_z_um = extract_scaling_metadata(file_path)

# Calculate anisotropy to rescale across the Z-axis (ratio of Z-resolution to XY-resolution)
rescale_factor = scaling_z_um / np.mean([scaling_x_um , scaling_y_um])

# Load precomputed labels when available; otherwise predict and store them
nuclei_labels = load_precomputed_results_if_available(nuclei_labels_dir, filename, results_type="nuclei_labels")

if nuclei_labels is not None:
    print(f"Predictions already calculated for: {filename} ...loading")
    # If nuclei_labels are loaded from disk, filter by MIN_MAX_NUCLEI_VOLUME if specified
    if MIN_MAX_NUCLEI_VOLUME is not None:
        from utils import _keep_objects_in_size_range  # safe in notebook
        nuclei_labels = _keep_objects_in_size_range(nuclei_labels, MIN_MAX_NUCLEI_VOLUME)

    # Add the prediction to the napari viewer
    viewer.add_labels(nuclei_labels)
    
else:
    # Predict nuclei labels using CellposeSAM using anisotropy correction
    nuclei_labels = predict_nuclei_labels(img, rescale_factor, NUCLEI_CHANNEL, MIN_MAX_NUCLEI_VOLUME, visualize=True)
    # Create path for nuclei labels (used only when saving a newly computed prediction)
    nuclei_labels_path = nuclei_labels_dir / f"{filename}_nuclei_labels.tif"
    # Save the prediction
    tifffile.imwrite(nuclei_labels_path, nuclei_labels)

# Simulate cytoplasm based on input parameters
cytoplasm = simulate_cytoplasm(nuclei_labels, cytoplasm_thickness=CYTOPLASM_THICKNESS, nuclei_padding=NUCLEI_PADDING)
viewer.add_labels(cytoplasm)

Nuclei labels directory: raw_data\WT_organoids\nuclei_labels\WT_organoids
Predictions already calculated for: RY20240607 4541 BCM 20X Hoechst bcat_GaMAlexa555 YAP_GaRAlexa647 UEA1_FITC 2 ...loading


<Labels layer 'cytoplasm' at 0x1ed8b0ff4f0>

### Compute YAP nuclei-to-cytoplasm ratio table
Extract intensity and volume features for nuclei and simulated cytoplasm, then compute per-label ratios.

In [47]:
# Automatically find the YAP channel index in MARKERS and extract the corresponding stack from img
yap_channel_index = next(i for i, marker in enumerate(MARKERS) if "YAP" in marker[0])

#Extract regionprops from filtered nuclei and generated-cytoplasm labels
props_nuclei = regionprops_table(label_image=nuclei_labels, intensity_image=img[yap_channel_index], properties=["label", "intensity_mean", "area"])
props_cytoplasm = regionprops_table(label_image=cytoplasm, intensity_image=img[yap_channel_index], properties=["label", "intensity_mean", "area"])


# Transform regionprops_table into a Dataframe to process it using Pandas
df_nuclei = pd.DataFrame(props_nuclei)
df_cytoplasm = pd.DataFrame(props_cytoplasm)

# Renaming columns
df_nuclei.rename(columns={'intensity_mean': 'intensity_mean_nuclei', 'area': 'volume_nuclei'}, inplace=True)
df_cytoplasm.rename(columns={'intensity_mean': 'intensity_mean_cyto', 'area': 'volume_cyto'}, inplace=True)

# Merge dataframes on label
merged_df = pd.merge(df_nuclei, df_cytoplasm, on='label')

# Calculate nuclei/cytoplasm ratio of mean intensity of YAP signal
merged_df['nuclei_cyto_ratio'] = merged_df['intensity_mean_nuclei'] / merged_df['intensity_mean_cyto']

# Add extracted ID info to the dataframe
merged_df.insert(0, 'experiment_id', experiment_id)
merged_df.insert(1, 'mouse_id', mouse_id)
merged_df.insert(2, 'treatment_id', treatment_id)
merged_df.insert(3, 'replica_id', replica_id)

merged_df.head(5)

,experiment_id,mouse_id,treatment_id,replica_id,label,intensity_mean_nuclei,volume_nuclei,intensity_mean_cyto,volume_cyto,nuclei_cyto_ratio
0,RY20240607,4541,BCM,2,1,1.410995,382.0,0.559367,758.0,2.522486
1,RY20240607,4541,BCM,2,2,12.018772,3143.0,6.930233,817.0,1.734252
2,RY20240607,4541,BCM,2,3,8.088129,556.0,7.434211,152.0,1.087961
3,RY20240607,4541,BCM,2,4,20.839627,2251.0,21.013699,365.0,0.991716
4,RY20240607,4541,BCM,2,5,17.625095,1315.0,13.800000,80.0,1.277181


### Query a specific label
Inspect per-label metrics for one nucleus to validate segmentation and intensity outputs.

Input needed from researcher: replace the label id with the nucleus of interest.

In [48]:
# Explore particular label results (i.e. 541)
label = 541

merged_df[merged_df['label']==label]

,experiment_id,mouse_id,treatment_id,replica_id,label,intensity_mean_nuclei,volume_nuclei,intensity_mean_cyto,volume_cyto,nuclei_cyto_ratio
538,RY20240607,4541,BCM,2,541,8.764566,841.0,5.248082,391.0,1.670051


### Inspect nuclei volume distribution
Review label volumes with a histogram to calibrate size-based filtering thresholds.

Input needed from researcher: inspect outliers and adjust `MIN_MAX_NUCLEI_VOLUME` if needed.

In [49]:
# Explore nuclei predicted labels by Cellpose to make an informed decision on the size-based exclusion of the predicted labels
# Create histogram
fig = px.histogram(merged_df, x="volume_nuclei", nbins=300, labels={"volume_nuclei": "Volume"}, title="Histogram of volumes of Cellpose predicted labels")

# Show plot
fig.show()

### Visualize measurements on nuclei labels
Project quantitative outputs back onto the nuclei labels to quickly inspect spatial patterns.

Input needed from researcher: choose the dataframe column to display and the colormap that best highlights your biological contrast.

In [50]:
map_df_column_to_labels(nuclei_labels, merged_df, value_column="volume_nuclei", colormap="viridis", visualize=True)
map_df_column_to_labels(nuclei_labels, merged_df, value_column="nuclei_cyto_ratio", colormap="inferno", visualize=True)

In [ ]:
from skimage.segmentation import relabel_sequential
from skimage.morphology import remove_small_objects

def segment_organoids_from_cp_labels(nuclei_labels, cytoplasm_thickness):
    
    """Segment whole organoids from input image using individual Cellpose nuclei labels as starting point"""

    # Generate cytoplasm labels via dilation
    cytoplasm_labels = cle.dilate_labels(nuclei_labels, radius=cytoplasm_thickness)

    # Maximum projection across z-axis to flatten 3D labels into 2D space
    mip_labels = np.max(cytoplasm_labels, axis=0)

    # Merge touching labels to establish first organoid entities
    merged_mip = cle.merge_touching_labels(mip_labels)

    # Dilation-Erosion cycle to close holes
    dilated_labels = cle.dilate_labels(merged_mip, radius=5)
    eroded_labels = cle.erode_labels(dilated_labels, radius=1)

    #TODO: Set min_size based on population wide stats of nuclei size
    # Pull from GPU in order to filter out small disconnected components and relabel using skimage
    org_labels = cle.pull(eroded_labels)
    org_labels = remove_small_objects(org_labels, min_size=15000)

    #Relabel starting from 1
    organoid_labels = relabel_sequential(org_labels)[0]

    return organoid_labels

In [52]:
organoid_labels = segment_organoids_from_cp_labels(nuclei_labels, cytoplasm_thickness=1)
organoid_mask = organoid_labels > 0
viewer.add_image(organoid_mask, opacity=0.5)

<Image layer 'organoid_mask' at 0x1ed997fba90>

In [54]:
# Set to 0 any nuclei_labels that does not sit inside the organoid_mask

# Ensure organoid_mask is a boolean mask and nuclei_labels has same spatial shape
# nuclei_labels: (z, y, x) or (y, x), organoid_mask: (z, y, x) or (y, x)
# We keep only those nuclei_labels voxels that are inside the organoid_mask.

# Match dimensionality so organoid_mask can be broadcast over nuclei_labels if necessary

# If organoid_mask is 2D and nuclei_labels is 3D, expand organoid_mask to 3D
if organoid_mask.ndim == 2 and nuclei_labels.ndim == 3:
    mask = np.broadcast_to(organoid_mask, nuclei_labels.shape)
elif organoid_mask.shape == nuclei_labels.shape:
    mask = organoid_mask
else:
    raise ValueError(f"Shape mismatch: organoid_mask shape {organoid_mask.shape} vs nuclei_labels shape {nuclei_labels.shape}")

nuclei_labels_filtered = nuclei_labels.copy()
nuclei_labels_filtered[~mask] = 0

map_df_column_to_labels(nuclei_labels_filtered, merged_df, value_column="volume_nuclei", colormap="viridis", visualize=True)
map_df_column_to_labels(nuclei_labels_filtered, merged_df, value_column="nuclei_cyto_ratio", colormap="inferno", visualize=True)